Matching Frames to Timestamps

In [ ]:
# Some basics:
import cv2
import math
import os
import pandas as pd


# Declare filenames
ROOT_DATA_DIR = (r"~\Data\Eyetracking_02_Data_Trials") # folder where the Input data sits
DATA_DIR_OUTPUT =  (r"~\Data\Eyetracking_03_Matched_frames")
VIDEO_DIR = (r"~\Lib")
MOVIE_STAMPS_DIR = (r'~\Data\Eyetracking_01_Preprocessed_Data')


# Define a dictionary to map movie labels to their corresponding video files
movie_files = {
    "movie_01": "Charite.mp4", #3724 frames
    "movie_02": "Ziemlich_Beste_Freunde.mp4", #3530
    "movie_03": "High_Seas.mp4", #3807
    "movie_04": "Biohackers.mp4", #3142
    "movie_05": "Downton_Abbey.mp4", #2842
    "movie_06": "New_Amsterdam.mp4" #3453 
}

# Define the directory containing the participant subfolders
participant_directory = ROOT_DATA_DIR

# Define the column names for each type of eyetracking data
fixation_columns = ['tStart', 'tEnd', 'duration', 'xAvg', 'yAvg', 'pupilAvg', 'Movie']
saccade_columns  = ['tStart', 'tEnd', 'duration', 'xStart', 'yStart', 'xEnd', 'yEnd', 'ampDeg', 'vPeak', 'Movie']
sample_columns   = ['tSample', 'LX', 'LY', 'LPupil', 'RX', 'RY', 'RPupil', 'Movie']


# Define an empty list to store the encountered errors
errors = []


# Loop through each participant subfolder
for participant_folder in os.listdir(participant_directory):
    # Construct the full path to the participant subfolder
    participant_folder_path = os.path.join(participant_directory, participant_folder)
    print('Participant Directory:', participant_folder_path)

    # Check if the subfolder is a directory
    if os.path.isdir(participant_folder_path):

        # Loop through each eyetracking data file for the participant
        for eyetracking_file in os.listdir(participant_folder_path):

            # Specify the output file path and name
            output_file_path = os.path.join(DATA_DIR_OUTPUT, participant_folder)
            output_file = os.path.join(output_file_path, f'{eyetracking_file[:-5]}_matched_frames.csv')
            # '%s\%s_%s_Block_%s_Order_%s_%s.xlsx' % (DATA_OUTPUT, NUM_FileStart, , BLOCK, ORDER, movie_num)

            if not os.path.exists(output_file_path):
                os.mkdir(output_file_path)

            if os.path.exists(output_file):
                #print(f"Output file already exists. Moving on to the next file.")
                continue  # Skip to the next iteration of the loop

            # Extract the information from the eyetracking file name
            print(eyetracking_file)
            file_parts = eyetracking_file[:-4].split("_")
            participant = file_parts[0]
            block_number = int(file_parts[-5])
            order_number = int(file_parts[-3])
            eyetracking_type = file_parts[-7]

            # Determine the column structure based on the eyetracking type
            if eyetracking_type == "Fixation":
                eyetracking_columns = fixation_columns
            elif eyetracking_type == "Saccade":
                eyetracking_columns = saccade_columns
            elif eyetracking_type == "Samples":
                eyetracking_columns = sample_columns
            else:
                print(f"Unknown eyetracking type in file: {eyetracking_file}")
                continue

            # Construct the full path to the eyetracking file
            eyetracking_file_path = os.path.join(participant_folder_path, eyetracking_file)

            # Load the eyetracking data into a DataFrame
            try:
                eyetracking_data = pd.read_excel((eyetracking_file_path))

                # Check if the eyetracking data is empty
                if eyetracking_data.empty:
                    errors.append((eyetracking_file_path, "Eyetracking data is empty"))
                    continue

                # Extract the desired columns from the eyetracking data if they exist in the file
                eyetracking_columns_present = [col for col in eyetracking_columns if col in eyetracking_data.columns]
                eyetracking_data = eyetracking_data[eyetracking_columns]

                # Check if the movie column exists in the eyetracking data
                if 'Movie' not in eyetracking_data.columns:
                    errors.append((eyetracking_file_path, "'Movie' column not found in eyetracking data"))
                    continue
                movie_label = eyetracking_data['Movie'].values[0]

                # Construct the video file name based on the extracted information
                video_file = movie_files[movie_label]
                video_path = os.path.join(VIDEO_DIR, video_file)
            
            except Exception as e:
                errors.append((eyetracking_file_path, str(e)))
                continue

            # except KeyError:
            #     print(f"Error: Movie label '{movie_label}' is not found in the movie_files dictionary.")
            #     continue
            # except FileNotFoundError:
            #     print(f"Error: Video file '{video_file}' is not found in the specified directory.")
            #     continue

            # Open the video file
            cap = cv2.VideoCapture(video_path)
            frame_rate = cap.get(cv2.CAP_PROP_FPS)

            # Calculate the frame timestamps
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            length_ms = int((total_frames / frame_rate) * 1000)
            #print("Video Length:", length_ms, "ms")
            frame_duration = 1.0 / frame_rate
            frame_timestamps = [frame_idx * (frame_duration * 1000) for frame_idx in range(total_frames)]

            # Release the video capture object
            cap.release()
            
            for movie_subdir in os.listdir(MOVIE_STAMPS_DIR):
                
                # Set the path to the directory for this subdirectory
                movie_subdir_path = os.path.join(MOVIE_STAMPS_DIR, movie_subdir)

                # Check if the subdir_path is a directory and does not end with '_02' # TO DO: Add Retrieval files. And extraction code
                if os.path.isdir(movie_subdir_path) and movie_subdir.startswith(participant) and not movie_subdir.endswith('_02'):
                    #print('Movie Subdirectory:', movie_subdir_path)

                    # Scan the subdirectory for files
                    file_list = os.listdir(movie_subdir_path)

                    movie_timestamps_files = [file_name for file_name in file_list if file_name.endswith('Movie_Timestamps.xlsx')]
                    movie_file_path = os.path.join(movie_subdir_path, movie_timestamps_files[0])
                    #print('movie_file_path=',movie_file_path)  
                            
                    # Load the files containing the start and end time points
                    DF_ts_movie = pd.read_excel((movie_file_path), index_col=0)
                    
                    # Check if the movie timestamps file exists
                    if not movie_timestamps_files:
                        errors.append(eyetracking_file_path,f"Error: Movie timestamps file not found in subdirectory '{file_list}'. Skipping...")
                        continue


            # Match eyetracking timestamps with frame timestamps
            if 'tSample' in eyetracking_data.columns:
                #print('Samples being matched...')
                matched_frames_start = []  # Initialize the list outside the loop
                
                for _, row in eyetracking_data.iterrows():
                    movie_num = row['Movie']
                    
                    # find movie starting time based on BLOCK and ORDER number. Cant use movie_num, as the names repeat
                    t_start_movie = float(DF_ts_movie.loc[(DF_ts_movie['Block'] == block_number) & (DF_ts_movie['Order'] == order_number), 't_start(ms)'].values[0])
                    t_start_eye = row['tSample']

                    abs_time_start = t_start_eye - t_start_movie
                    closest_frame_start = round(min(range(total_frames), key=lambda x: abs(frame_timestamps[x] - abs_time_start)))
                    matched_frames_start.append(closest_frame_start)

                # Add the matched_frames columns to the eyetracking_data dataframe
                eyetracking_data['matched_frame_IDx'] = matched_frames_start

                # Save the modified eyetracking data to a new file
                eyetracking_data.to_csv(output_file)
                print('Block', block_number, 'Order', order_number)
                print('File Saved')
                

                # # Update BLOCK and ORDER for the next iteration of file.
                # ORDER += 1
                # if ORDER > 6:
                #     ORDER = 1
                #     BLOCK += 1
                #     if BLOCK > 3:
                #         BLOCK = 1

            else:
            # Match eyetracking timestamps with frame timestamps
                matched_frames_start = []
                matched_frames_end = []

                # DF_ts_movie
                #print(DF_ts_movie.head())

                for _, row in eyetracking_data.iterrows():
                    # movie_num = row['Movie']
                    
                    t_start_movie = float(DF_ts_movie.loc[(DF_ts_movie['Block'] == block_number) & (DF_ts_movie['Order'] == order_number), 't_start(ms)'].values[0])
                    t_end_movie = float(DF_ts_movie.loc[(DF_ts_movie['Block'] == block_number) & (DF_ts_movie['Order'] == order_number), 't_end(ms)'].values[0])
                    
                    t_start_eye = row['tStart']
                    t_end_eye = row['tEnd']
                    
                    abs_time_start = t_start_eye - t_start_movie
                    closest_frame_start = round(min(range(total_frames), key=lambda x: abs(frame_timestamps[x] - abs_time_start)))
                    matched_frames_start.append(closest_frame_start)

                    abs_time_end = t_end_eye - t_start_movie
                    closest_frame_end = round(min(range(total_frames), key=lambda x: abs(frame_timestamps[x] - abs_time_end)))
                    matched_frames_end.append(closest_frame_end)

                    # print('t_start_movie',t_start_movie)
                    # print('t_end_movie',t_end_movie)
                    # print('t_start_eye',t_start_eye)
                    # print('t_end_eye',t_end_eye)

                    # print('abs_time_start',abs_time_start)
                    # print('abs_time_end',abs_time_end)

                # Specify the output file path and name
                # Add the matched frames to the eyetracking data
                eyetracking_data['matched_frame_IDx_start'] = matched_frames_start
                eyetracking_data['matched_frame_IDx_end'] = matched_frames_end

                # Save the modified eyetracking data to a new file
                eyetracking_data.to_csv(output_file)
                print('Block', block_number, 'Order', order_number)
                print('File Saved')

    # Print the error summary
    if errors:
        print("Error Summary:")
        for error_file, error_message in errors:
            print(f"File: {error_file} - Error: {error_message}")
    else:
        print("Processing completed without errors.")

if participant_folder == os.listdir(participant_directory)[-1]:
    print('Processed all subdirectories.')
    winsound.MessageBeep()
    break
# print("Participant ", participant, "Type", eyetracking_type, "Block", block_number, "Order", order_number, "Done!")

#53.46
#325.44


